<img src="https://raw.githubusercontent.com/reflective-org/AAAR-2026-Aerosol-Tutorial/520b58ef28d16cd1fa2a7bdbf7573365fd88aaec/assets/banner-level1.png" width="100%" alt="AAAR 2026 photochemical box modeling tutorial. Level 1 of 3: Volatility. When does a vapor make particles grow? Supported by Reflective.">

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/reflective-org/AAAR-2026-Aerosol-Tutorial/blob/main/notebooks/01_volatility.ipynb)

A box of air holds a seed aerosol: 2000 particles per cm³ around 50 nm, 30 % of their mass a non-volatile organic. One made-up organic vapor, the *surrogate*, is produced at a steady rate. How much of it ends up in the particles depends on its **saturation concentration c\***. You can also add sulfuric acid (H₂SO₄), which is effectively non-volatile, switch on new-particle formation (nucleation), and watch coagulation merge particles. There is no gas-phase chemistry yet; that comes in Level 2. The model is [TOMAS-JAX](https://github.com/reflective-org/TOMAS-JAX): TOMAS sectional aerosol microphysics (80 size bins from 1.7 nm to 17.5 µm; the web app uses 40) with kinetic gas–particle partitioning of organics. This notebook is the "look under the hood" companion to Level 1 of the tutorial web app.

> <font color="#8B8D8E"><b>HOW TO USE IT</b></font> &nbsp; Run the cells from top to bottom (Shift+Enter). Boxes labelled <font color="#A47F52"><b>TRY THIS</b></font> name a variable to change: edit it, run that cell again, then the plot cells below it. On Colab those cells also show a title and an input box; editing the box changes the code. All plots are interactive: hover for values, drag to zoom, double-click to reset, click a legend entry to hide or show a line.

<details>
<summary><b>Run times</b></summary>

One 12-hour Level 1 run takes about 6 s on a laptop and roughly twice that on Colab. The first run in a session also compiles the model with JAX (about 2 s for Level 1; Levels 2 and 3, with gas chemistry, need about 5 s on a laptop, more on Colab). Switching a process such as nucleation on or off compiles once more. The whole notebook runs in about a minute and a half once installed.

</details>

## Setup

> <font color="#8B8D8E"><b>SETUP</b></font> &nbsp; Run the next two cells once per session.
>
> - **Colab:** the first cell installs the box model and the tutorial helpers (1–2 minutes). Red pip
>   "dependency conflicts" lines are expected. If a cell asks you to restart, click
>   *Runtime › Restart session*, then *Runtime › Run all*.
> - **Your own computer:** install the repository first ([README](https://github.com/reflective-org/AAAR-2026-Aerosol-Tutorial/blob/main/README.md#run-on-your-own-computer));
>   the first cell then does nothing.
> - `at.setup()` downloads the TUV-x photolysis data once (about 5 MB) and prints the versions.

In [ ]:
# Colab: installs the tutorial helpers and the box model (first time: 1-2 minutes).
# Red pip "dependency conflicts" lines on Colab are expected and do not matter here.
import importlib.metadata
import sys

HELPERS_VERSION = "0.3.0"   # the aaar_tutorial version these notebooks need


def _older(version):
    return tuple(map(int, version.split("."))) < tuple(map(int, HELPERS_VERSION.split(".")))


try:
    installed = importlib.metadata.version("aaar-tutorial")
except importlib.metadata.PackageNotFoundError:
    installed = None
if installed is None or _older(installed):
    if "google.colab" not in sys.modules:
        raise ModuleNotFoundError(
            f"This notebook needs the tutorial helpers {HELPERS_VERSION}, found {installed or 'none'}. Pick the "
            "kernel of the repository's .venv, update it (git pull, then uv sync or pip install -e .), then "
            "restart the kernel.")
    %pip install -q "aaar-tutorial @ git+https://github.com/reflective-org/AAAR-2026-Aerosol-Tutorial@main"
    importlib.invalidate_caches()           # so the next cell finds the new packages
loaded = getattr(sys.modules.get("aaar_tutorial"), "__version__", None)
if loaded is not None and _older(loaded):
    raise RuntimeError("This session still has older tutorial helpers loaded. Colab: Runtime > Restart "
                       "session, then Runtime > Run all. Locally: Kernel > Restart Kernel.")

In [ ]:
import aaar_tutorial as at
from aaar_tutorial import config, plots

at.setup()

## Learning goals

> <font color="#8B8D8E"><b>LEARNING GOALS</b></font> &nbsp; After this notebook you can:
>
> 1. Say what the saturation concentration c\* is and predict a vapor's particle-phase fraction with the absorptive-partitioning equation.
> 2. Tell a semivolatile vapor from a non-volatile one (H₂SO₄) by how they grow a size distribution.
> 3. Recognise coagulation on its own: particle number falls, mass stays the same.
> 4. Explain why nucleation needs a lot of H₂SO₄ and, in this model, cold air.
> 5. Change any model input from Python, run the model and compare runs in interactive plots.

## 1. The physics in a few equations

**Saturation concentration.** c\* (µg m⁻³) is the *effective* saturation concentration (Donahue et al., 2006): the saturation concentration of the pure compound, C⁰, times its activity coefficient in the organic aerosol. This model treats the organic phase as an ideal mixture (activity coefficient 1), so here c\* is simply the gas-phase concentration in equilibrium over the pure compound. The lower c\*, the less volatile. Rough classes (Donahue et al., 2012): below 0.3 µg m⁻³ low or extremely low volatility (LVOC, ELVOC), 0.3–300 µg m⁻³ semivolatile (SVOC), above that intermediate volatility (IVOC).

> <font color="#8B8D8E"><b>KEY IDEA</b></font> &nbsp; **Absorptive partitioning** (Pankow, 1994). At equilibrium the fraction of a compound in the particles is
>
> $$F_p = \frac{1}{1 + c^*/C_{OA}},$$
>
> where $C_{OA}$ is the organic aerosol mass the compound can dissolve into: here the seed's non-volatile organic (reported as POA) plus the SOA itself. Sulfate and water do not absorb organics in this model. <mark>A vapor with $c^* = C_{OA}$ is split half and half.</mark>


In [ ]:
# No model run: the partitioning equation for each c* class, with C_OA on the slider.
from aaar_tutorial import plots      # repeated, so this cell also runs on its own

plots.partitioning(highlight_cstar=1, highlight_label="the surrogate")

> <font color="#A47F52"><b>EXPLORE</b></font> &nbsp; Drag the slider (or press ▶) to change $C_{OA}$, the organic aerosol mass the vapors can dissolve into. Each bar is one volatility class: green is its share in the particles, blue its share in the gas. The dashed line marks $c^* = C_{OA}$; classes to its left are mostly in the particles. Ambient $C_{OA}$ ranges from well below 1 µg m⁻³ in clean air and 10–20 µg m⁻³ or more in polluted cities to hundreds of µg m⁻³ in wildfire smoke, and more close to the fire. Here $C_{OA}$ is a knob; in the model it is the seed's POA plus the SOA that forms, so it grows during a run.
>
> <font color="#A47F52"><b>TRY THIS</b></font> &nbsp; At which $C_{OA}$ is the surrogate (c\* = 1, outlined) 90 % in the particles? Compare with question 1 at the end.

**Condensation and evaporation.** The model does not jump to equilibrium. Each size bin $k$ (diameter $D_{p,k}$, number $N_k$) exchanges mass with the gas at a finite rate:

$$\frac{dC_{p,k}}{dt} = 2\pi\, D_g\, D_{p,k}\, F(\mathrm{Kn}_k, \alpha)\, N_k \left(C_g - C^{eq}_k\right), \qquad C^{eq}_k = c^*\, X_k \exp\!\left(\frac{4\sigma M}{R T \rho D_{p,k}}\right).$$

$C_g$ is the vapor concentration, $D_g$ its diffusivity and $F$ the Fuchs–Sutugin transition-regime correction. $F$ depends on the Knudsen number $\mathrm{Kn}_k = 2\lambda/D_{p,k}$ (λ is the mean free path of the vapor molecules) and on the mass accommodation coefficient α, the fraction of molecules that stick when they hit a particle. This model uses α = 1: every molecule that hits a particle sticks, the fastest uptake possible. $X_k$ is the compound's share of the organic mass in bin $k$ (Raoult's law, by mass in this model) and the exponential is the Kelvin effect, which makes evaporation easier from small particles. $C_g > C^{eq}_k$ means condensation, $C_g < C^{eq}_k$ evaporation. H₂SO₄ has $c^* \approx 0$, so it only condenses.

**Temperature.** c\* is given at 298 K and follows Clausius–Clapeyron, with $\Delta H_{vap} = 131 - 11\log_{10} c^*$ kJ mol⁻¹ (Epstein et al., 2010):

$$c^*(T) = c^*(298\,\mathrm{K}) \exp\!\left[-\frac{\Delta H_{vap}}{R}\left(\frac{1}{T} - \frac{1}{298\,\mathrm{K}}\right)\right].$$

**Coagulation.** Particles collide and stick. For one size, $dN/dt = -\tfrac{1}{2} K N^2$: number falls, mass is conserved, and the relative loss rate $\tfrac{1}{2} K N$ grows with $N$.

**Nucleation.** H₂SO₄ and water form new clusters in the smallest bin (1.7 nm) at a rate $J$ that rises steeply with [H₂SO₄] and falls steeply with temperature. In this tutorial it is pure H₂SO₄–H₂O nucleation (no NH₃, ions or organics; ASSUMPTIONS.md A13). It is off in the preset.

> <font color="#8B8D8E"><b>THE BOX</b></font> &nbsp; The box has no dilution, deposition or walls. The model steps the aerosol (coagulation, condensation, partitioning, water uptake) every minute and saves results every 10 minutes.


## 2. What is in the preset?

`at.describe` prints every input of a scenario. Each one can be changed when you run the model, e.g. `at.run("level1_volatility", surrogate={"cstar_ugm3": 10})`. A misspelled input gives an error that suggests the right spelling.

In [ ]:
at.describe("level1_volatility")

The inputs that matter here:

| Input | Preset | Meaning |
|---|---|---|
| `surrogate={"cstar_ugm3": ...}` | 1 | c\* of the surrogate at 298 K (µg m⁻³) |
| `surrogate={"production_ugm3_per_h": ...}` | 0.5 | vapor production (µg m⁻³ h⁻¹); 0 switches the organic off |
| `h2so4_production_cm3_s=...` | 0 | H₂SO₄ production (molecules cm⁻³ s⁻¹); 0 is off |
| `processes={"nucleation": ...}` | False | new-particle formation |
| `processes={"coagulation": ...}` | True | coagulation |
| `seed={"number_cm3": ...}` | 2000 | seed particles per cm³ |
| `seed={"gmd_nm": ...}` | 50 | geometric mean diameter (GMD) of the seed mode, nm |
| `seed={"gsd": ...}` | 1.6 | geometric standard deviation (GSD): the width of the lognormal mode |
| `seed={"organic_mass_fraction": ...}` | 0.3 | non-volatile organic share of the seed mass (the rest is sulfate) |
| `environment={"temp_k": ..., "rh": ...}` | 298.15 K, 0.5 | temperature, relative humidity |
| `duration_h=...` | 12 | run length; must be a whole number of 5-minute steps |

A `seed={...}` dict changes only the keys you give; `seed={"modes": [...]}` sets several modes at once. At 0.5 µg m⁻³ h⁻¹ for 12 h the box makes 6 µg m⁻³ of organic vapor in total.

## 3. First look: the preset

`at.run` runs the model and shows a progress bar: about 6 s, plus a few seconds of compiling the first time. Set `USE_PRECOMPUTED = True` to load a stored result of the unchanged preset instead (instant). The experiments below always run the model.

In [ ]:
#@title Run the preset
# True: load the stored result of the unchanged preset (no model run)
USE_PRECOMPUTED = False  #@param {type:"boolean"}

if USE_PRECOMPUTED:
    base = at.load_example("level1_volatility")
else:
    base = at.run("level1_volatility")

for key, value in at.summary(base).items():
    print(f"{key:>16}  {value:.4g}")

**Size distribution.** Drag the time slider or press ▶. Grey dotted: the start; faint lines: every hour; bold: the selected time. The mode moves to larger sizes as the vapor condenses. Diameters are dry and reported at a density of 1400 kg m⁻³, so the 50 nm seed (placed at sulfate density) appears at about 54 nm. The zigzag from bin to bin is numerical: it comes from moving growing particles between fixed size bins. Each of the 80 bins holds particles √2 ≈ 1.4 times as heavy as the bin below (the web app uses 40, each twice as heavy).

In [ ]:
plots.size_distribution(base)

**Banana plot.** The same distribution over time as a heat map: growth shows as the band rising.

In [ ]:
plots.banana(base)

**Volatility.** Green: surrogate mass in the particles; blue: still in the gas. The dashed line is $C_{OA}$ (SOA + POA). With one surrogate there is one bar, at c\* = 1 µg m⁻³. Move the slider back to the first hour: $C_{OA}$ is then only the seed's 0.19 µg m⁻³ of organic, so most of the vapor stays in the gas. As SOA builds up, $C_{OA}$ grows and pulls more vapor in. The gas levels off near 1 µg m⁻³: once the particle organic is mostly surrogate ($X \approx 1$), the vapor over it is close to c\* itself.

In [ ]:
plots.volatility(base)

In [ ]:
plots.totals(base, keys=("N_cm3", "SOA_ugm3", "dry_ugm3"))

> <font color="#858F6A"><b>WHAT TO NOTICE</b></font> &nbsp; SOA rises steadily, particle number falls by about 10 % (coagulation), and dry mass grows by the SOA.

## 4. Under the hood: how close is the model to equilibrium?

For one compound with total mass $C_{tot} = C_p + C_g$ and non-volatile organic $C_{nv}$, the partitioning equation with $C_{OA} = C_{nv} + C_p$ is a quadratic in the particle mass $C_p$:

$$C_p = \tfrac{1}{2}\left[-b + \sqrt{b^2 + 4\, C_{tot} C_{nv}}\right], \qquad b = C_{nv} + c^* - C_{tot}.$$

The cell below compares this equilibrium with the model at the end of a run. `at.cstar_at(c*, T)` applies the temperature formula above (the preset box is at 298.15 K, so c\* is about 3 % above its 298 K value). Read the code: everything comes from the `RunResult` the model returns. The plot shows the surrogate in the particles (SOA) and in the gas (`org_gas_ugm3`).

In [ ]:
import numpy as np


def pankow_particle_mass(c_tot, c_star, c_nv):
    """Equilibrium particle-phase mass (µg/m³) of one compound; c_nv = non-volatile organic."""
    b = c_nv + c_star - c_tot
    return 0.5 * (-b + np.sqrt(b**2 + 4 * c_tot * c_nv))


def compare_with_pankow(result):
    """Print the model's SOA at the end of a run next to the equilibrium value."""
    soa = at.series(result, "SOA_ugm3")[-1]
    gas = at.series(result, "org_gas_ugm3")[-1]
    poa = at.series(result, "POA_ugm3")[-1]
    c_star = at.cstar_at(result.scenario["surrogate"]["cstar_ugm3"], result.scenario["environment"]["temp_k"])
    expected = pankow_particle_mass(soa + gas, c_star, poa)
    print(f"{result.scenario['name']}: c*(T) = {c_star:.3g} µg/m³, total organic = {soa + gas:.3g}, "
          f"POA = {poa:.3g} -> SOA model = {soa:.3g}, equilibrium = {expected:.3g} µg/m³")


compare_with_pankow(base)
plots.time_series(base, ["SOA_ugm3", "org_gas_ugm3"], title="Surrogate organic: in the particles and in the gas")

> <font color="#858F6A"><b>WHAT TO NOTICE</b></font> &nbsp; The model ends about 1.4 % below equilibrium. Vapor made in the last few minutes has not condensed yet, and the Kelvin effect raises the equilibrium vapor over the smaller particles.

## 5. Experiment 1: sweep c\*

Three runs that differ only in c\*. Each takes about 6 s.

> <font color="#A47F52"><b>TRY THIS</b></font> &nbsp; Change `CSTAR_VALUES`, e.g. to `[0.1, 1, 10]` or `[1, 10, 1000]` (the web app's slider goes from 0.001 to 10 000).

In [ ]:
#@title Choose the c* values and run
# c* in µg/m³ at 298 K
CSTAR_VALUES = [0.01, 1, 100]  #@param {type:"raw"}

cstar_runs = [at.run("level1_volatility", surrogate={"cstar_ugm3": c}, name=f"c* = {c:g}")
              for c in CSTAR_VALUES]
for r in cstar_runs:
    compare_with_pankow(r)

In [ ]:
plots.time_series(cstar_runs, ["SOA_ugm3", "org_gas_ugm3"])

In [ ]:
plots.compare_size_distributions(cstar_runs)

In [ ]:
plots.time_series(cstar_runs, "gmd_nm", title="Number geometric mean diameter")

> <font color="#858F6A"><b>WHAT TO NOTICE</b></font> &nbsp; The c\* = 0.01 vapor condenses almost completely, like a non-volatile one. At c\* = 100 nearly everything stays in the gas, because $C_{OA}$ never gets close to 100 µg m⁻³. Notice that c\* = 0.01 grows the particles much more in diameter than c\* = 1 although its SOA is only about 20 % higher. Compare the mass distributions below: <mark>a low-volatility vapor goes where the particle surface is and grows every particle, while a semivolatile one settles where the organic mass already is</mark> (the large particles) and is pushed off the small ones by the Kelvin effect.

In [ ]:
plots.compare_size_distributions(cstar_runs, quantity="mass")

In [ ]:
plots.volatility(cstar_runs[-1])   # the most volatile run: almost all gas

## 6. Experiment 2: H₂SO₄, the organic, both or neither

The preset has no H₂SO₄ (`h2so4_production_cm3_s = 0`). H₂SO₄ is effectively non-volatile: every molecule that reaches a particle stays. Below, four runs switch each vapor on or off. The first is the preset (organic only); with both vapors off, only coagulation changes the particles.

> <font color="#A47F52"><b>TRY THIS</b></font> &nbsp; Change `H2SO4_PRODUCTION` (the web app's slider spans 10³–10⁷ molec cm⁻³ s⁻¹).

In [ ]:
#@title Set the H₂SO₄ production and run the four cases
# molecules per cm³ per s
H2SO4_PRODUCTION = 1e6  #@param {type:"raw"}

print(f"H2SO4 production = {H2SO4_PRODUCTION * 98.08 / 6.022e23 * 1e12 * 3600:.2f} µg/m³/h "
      "(the organic: 0.5 µg/m³/h)")

both = at.run("level1_volatility", h2so4_production_cm3_s=H2SO4_PRODUCTION, name="organic + H₂SO₄")
h2so4_only = at.run("level1_volatility", h2so4_production_cm3_s=H2SO4_PRODUCTION,
                    surrogate={"production_ugm3_per_h": 0}, name="H₂SO₄ only")
no_vapor = at.run("level1_volatility", surrogate={"production_ugm3_per_h": 0}, name="no vapor")

vapor_runs = [base, both, h2so4_only, no_vapor]
vapor_labels = ["organic (preset)", "organic + H₂SO₄", "H₂SO₄ only", "no vapor"]

In [ ]:
plots.compare_size_distributions(vapor_runs, labels=vapor_labels)

In [ ]:
plots.totals(vapor_runs, labels=vapor_labels, keys=("N_cm3", "SO4_ugm3", "SOA_ugm3", "dry_ugm3"))

In [ ]:
plots.composition(both)

> <font color="#858F6A"><b>WHAT TO NOTICE</b></font> &nbsp;
>
> - **H₂SO₄ only** adds about as much mass per hour as the organic, but all of it ends up in the particles, and the whole distribution moves to larger sizes. Its c\* is effectively zero.
> - **Organic + H₂SO₄**: the sulfate does not dissolve the organic (only organics absorb organics here), so SOA is almost the same as in the preset.
> - **No vapor**: number falls slowly by coagulation and dry mass stays exactly constant. At 2000 cm⁻³ coagulation is slow; the next experiment speeds it up.
> - The two H₂SO₄ runs lose fewer particles than the other two. See question 5 below.

## 7. Experiment 3: seed number and size

Coagulation scales with $N^2$. With ten times more seed particles, they collide a hundred times as often in total, and each particle ten times as often.

> <font color="#A47F52"><b>TRY THIS</b></font> &nbsp; Change `SEED_NUMBER` (the web app allows 10 to 10⁵ cm⁻³). `seed={"number_cm3": ...}` changes only the number; the diameter and GSD keep their preset values.

In [ ]:
#@title Choose the seed number (coagulation only)
# particles per cm³ (preset: 2000)
SEED_NUMBER = 2e4  #@param {type:"raw"}

dense_no_vapor = at.run("level1_volatility", seed={"number_cm3": SEED_NUMBER},
                        surrogate={"production_ugm3_per_h": 0}, name=f"{SEED_NUMBER:g} cm⁻³, no vapor")
for r in (no_vapor, dense_no_vapor):
    n = at.series(r, "N_cm3")
    print(f"{r.scenario['name']}: N {n[0]:.0f} -> {n[-1]:.0f} cm⁻³ ({100 * (1 - n[-1] / n[0]):.0f} % lost in 12 h)")

plots.time_series([no_vapor, dense_no_vapor], ["N_cm3", "dry_ugm3"], log_y=True,
                  labels=["2000 cm⁻³", f"{SEED_NUMBER:g} cm⁻³"], title="Coagulation only (no vapor)")

Now keep the organic on and change the seed. More (or bigger) seed particles bring more surface and more non-volatile organic to dissolve into.

> <font color="#A47F52"><b>TRY THIS</b></font> &nbsp; Change `SEED_GMD_NM`, or delete the `#` in front of the pure-sulfate run in the list to add a seed without organic. Any run you add to the list appears in the plots.

In [ ]:
#@title Choose the seed size and run
# nm (preset: 50)
SEED_GMD_NM = 150  #@param {type:"raw"}

seed_runs = [
    base,
    at.run("level1_volatility", seed={"number_cm3": SEED_NUMBER}, name=f"{SEED_NUMBER:g} cm⁻³, 50 nm"),
    at.run("level1_volatility", seed={"gmd_nm": SEED_GMD_NM}, name=f"2000 cm⁻³, {SEED_GMD_NM:g} nm"),
    # at.run("level1_volatility", seed={"organic_mass_fraction": 0.0}, name="2000 cm⁻³, 50 nm, pure sulfate"),
]
seed_labels = ["2000 cm⁻³, 50 nm (preset)"] + [r.scenario["name"] for r in seed_runs[1:]]
for r in seed_runs:
    compare_with_pankow(r)

plots.time_series(seed_runs, ["SOA_ugm3", "POA_ugm3", "N_cm3"], labels=seed_labels, title="Seed number and size")

In [ ]:
plots.time_series(seed_runs, "gmd_nm", labels=seed_labels, title="Number geometric mean diameter")

> <font color="#858F6A"><b>WHAT TO NOTICE</b></font> &nbsp; More seed organic (POA) means a larger $C_{OA}$, so the c\* = 1 vapor starts condensing sooner and a little more of it ends up in the particles. The same SOA spread over more, or larger, particles changes their diameter less. In the dense run, though, coagulation removes about half of the particles and grows the survivors, which offsets most of that dilution: its GMD ends at 84 nm, not far below the preset's 98 nm.

## 8. Experiment 4: nucleation

Turn nucleation on together with H₂SO₄. The first run compiles once more (a few seconds) because a process changed. The model's nucleation is H₂SO₄–H₂O only, which is very slow at 298 K, so compare a warm and a cold box.

> <font color="#A47F52"><b>TRY THIS</b></font> &nbsp; Change `NUCLEATION_TEMPS_K`, e.g. `[280, 275, 270]`. Where does new-particle formation switch on?

In [ ]:
#@title Choose the temperatures and run
# K
NUCLEATION_TEMPS_K = [298.15, 265]  #@param {type:"raw"}

nucleation_runs = [
    at.run("level1_volatility", h2so4_production_cm3_s=H2SO4_PRODUCTION, processes={"nucleation": True},
           environment={"temp_k": t}, name=f"nucleation on, {t:g} K")
    for t in NUCLEATION_TEMPS_K
]
for t in NUCLEATION_TEMPS_K:
    print(f"surrogate c* at {t:g} K: {at.cstar_at(1.0, t):.2g} µg/m³")

In [ ]:
plots.time_series(nucleation_runs, ["N_cm3", "nucleation_rate_cm3_s", "h2so4_cm3"], log_y=True,
                  title="Nucleation: warm and cold box")

In [ ]:
plots.banana(nucleation_runs[-1])

> <font color="#858F6A"><b>WHAT TO NOTICE</b></font> &nbsp; In the cold run new particles appear at the bottom of the banana plot within the first hour and grow into the seed mode. Particle number rises above the seed's 2000 cm⁻³, then coagulation brings it down. At 298 K the nucleation rate stays far below 1 cm⁻³ s⁻¹, so no new particles show up. Note that the cold box also changes the organic: its c\* drops by a factor of several hundred, so it behaves almost like a non-volatile vapor and helps the new particles grow.

## 9. Your turn: run panel

> <font color="#A47F52"><b>YOUR TURN</b></font> &nbsp; Pick c\* and the vapor production, then press **Run model** (about 6 s). The plots compare your run with the preset. Nothing runs until you press the button.

In [ ]:
import ipywidgets as widgets

run_panel = widgets.interact_manual.options(manual_name="Run model")


@run_panel(
    cstar=widgets.FloatLogSlider(value=1, base=10, min=-3, max=4, step=0.5, readout_format=".3g",
                                 description="c* (µg/m³)", style={"description_width": "120px"}),
    production=widgets.FloatSlider(value=0.5, min=0, max=5, step=0.25,
                                   description="P_org (µg/m³/h)", style={"description_width": "120px"}),
)
def explore(cstar, production):
    print(f"Running c* = {cstar:g} µg/m³, P_org = {production:g} µg/m³/h ...")
    run = at.run("level1_volatility", progress=False, name="your run",
                 surrogate={"cstar_ugm3": cstar, "production_ugm3_per_h": production})
    compare_with_pankow(run)
    labels = ["preset: c* = 1, P = 0.5", f"your run: c* = {cstar:g}, P = {production:g}"]
    plots.compare_size_distributions([base, run], labels=labels).show()
    plots.time_series([base, run], ["SOA_ugm3", "org_gas_ugm3"], labels=labels).show()

## Questions to think about

> 1. A vapor with c\* = 10 µg m⁻³ meets 2 µg m⁻³ of organic aerosol. Roughly what fraction is in the particles at equilibrium? And in a polluted city with 20 µg m⁻³?
> 2. Why does the gas-phase surrogate level off near c\* in the preset, even though vapor is produced the whole time?
> 3. Why does the c\* = 0.01 vapor grow the particles' diameter so much more than the c\* = 1 vapor, when its SOA is only about 20 % larger?
> 4. In the run without vapor, number falls but dry mass stays constant. Which process is acting, and why is it so slow at 2000 cm⁻³?
> 5. H₂SO₄ at 10⁶ molec cm⁻³ s⁻¹ adds about the same mass per hour as the organic. Why does all of it end up in the particles? And why do the H₂SO₄ runs lose fewer particles to coagulation? (Hint: coagulation is fastest between particles of very different sizes. Compare the widths of the distributions.)
> 6. Nucleation did almost nothing at 298 K. What is missing from this model that boundary-layer nucleation in the real atmosphere depends on?

## Going further

- **Temperature.** The first cell below reruns the preset at 278 K. Use `at.cstar_at(1.0, 278)` to explain the extra SOA, then try a hot day (308 K). Use that cell as a starting point for your own experiments.
- **Seed composition.** `seed={"organic_mass_fraction": 0.0}` (pure sulfate) or `0.6`: how much does the absorbing seed organic matter once SOA has formed?
- **A vapor that is already there.** `surrogate={"initial_gas_ugm3": 5, "production_ugm3_per_h": 0}` starts with 5 µg m⁻³ of vapor and no source.
- **Everything the model returns.** `at.available_series(result)` lists the time series; `result.dndlog10dp_cm3` is the size distribution [time, bin]; `result.volatility` holds gas and particle mass per c\* class; `at.save_result(result, "my_run.json")` saves a run and `at.load_result` reads it back.
- **The raw model API.** `at.run` is a thin wrapper: it builds a `Scenario` and calls `tomas_jax.box.driver.run`. The last cell does the same by hand, with the preset's 40 size bins (`at.run` uses 80); [04_inside_the_model.ipynb](04_inside_the_model.ipynb) goes further.
- **The source.** Partitioning: [`tomas_jax/physics/soa_partitioning.py`](https://github.com/reflective-org/TOMAS-JAX/blob/87537364420e76cd50203049e4d74cb23660327d/tomas_jax/physics/soa_partitioning.py) and its [documentation](https://github.com/reflective-org/TOMAS-JAX/blob/87537364420e76cd50203049e4d74cb23660327d/docs/aaar/soa_partitioning.md); the aerosol step that combines all processes: [`docs/aaar/aerosol_step.md`](https://github.com/reflective-org/TOMAS-JAX/blob/87537364420e76cd50203049e4d74cb23660327d/docs/aaar/aerosol_step.md).
- **Assumptions.** Every simplification is listed in [ASSUMPTIONS.md](https://github.com/reflective-org/TOMAS-JAX/blob/87537364420e76cd50203049e4d74cb23660327d/ASSUMPTIONS.md). For this level: A6 (volatility classes and ΔH_vap), A9 (organics take up water like ammonium bisulfate), A13 (H₂SO₄-only nucleation), A14 (seed and box), AC1 (a small absorbing floor that stops a vapor condensing on organic-free particles).
- **Next: [02_limonene_soa.ipynb](02_limonene_soa.ipynb).** In Level 2 the vapor is no longer a knob: limonene chemistry makes hundreds of products, each with its own c\*. For now those c\* values are estimated from each product's elemental composition (Li et al., 2016), a placeholder until a curated table is in (ASSUMPTIONS.md A8).

In [ ]:
#@title Your own experiments
# Start here. Example: the preset in a colder box (c* falls with temperature).
cold = at.run("level1_volatility", environment={"temp_k": 278}, name="278 K")
compare_with_pankow(cold)
plots.time_series([base, cold], ["SOA_ugm3", "N_cm3"], labels=["298 K (preset)", "278 K"], title="Temperature")

The raw model API, without the tutorial helpers:

In [ ]:
from tomas_jax.box import driver, scenario

s = scenario.load_preset("level1_volatility").replace(surrogate={"cstar_ugm3": 10}, name="raw API, c* = 10")
print(s.surrogate)
raw = driver.run(s)
print(f"SOA after {raw.time_h[-1]:g} h: {raw.totals['SOA_ugm3'][-1]:.3g} µg/m³ (run time {raw.diagnostics['runtime_s']:.1f} s)")

### References

<details>
<summary>7 references (click to show)</summary>

- Donahue, N. M., Robinson, A. L., Stanier, C. O., and Pandis, S. N.: Coupled partitioning, dilution, and chemical aging of semivolatile organics, *Environ. Sci. Technol.*, 40, 2635–2643, 2006.
- Donahue, N. M., Kroll, J. H., Pandis, S. N., and Robinson, A. L.: A two-dimensional volatility basis set – Part 2: Diagnostics of organic-aerosol evolution, *Atmos. Chem. Phys.*, 12, 615–634, 2012.
- Dunne, E. M., et al.: Global atmospheric particle formation from CERN CLOUD measurements, *Science*, 354, 1119–1124, 2016.
- Epstein, S. A., Riipinen, I., and Donahue, N. M.: A semiempirical correlation between enthalpy of vaporization and saturation concentration for organic aerosol, *Environ. Sci. Technol.*, 44, 743–748, 2010.
- Li, Y., Pöschl, U., and Shiraiwa, M.: Molecular corridors and parameterizations of volatility in the chemical evolution of organic aerosols, *Atmos. Chem. Phys.*, 16, 3327–3344, 2016.
- Pankow, J. F.: An absorption model of gas/particle partitioning of organic compounds in the atmosphere, *Atmos. Environ.*, 28, 185–188, 1994.
- Zaveri, R. A., Easter, R. C., Shilling, J. E., and Seinfeld, J. H.: Modeling kinetic partitioning of secondary organic aerosol and size distribution dynamics, *Atmos. Chem. Phys.*, 14, 5153–5181, 2014.

</details>

---

<a href="https://www.reflective.org"><img src="https://raw.githubusercontent.com/reflective-org/AAAR-2026-Aerosol-Tutorial/520b58ef28d16cd1fa2a7bdbf7573365fd88aaec/assets/supported-by-reflective.png" width="294" alt="Supported by Reflective"></a>